- Documents module for data retrieval and processing workflows.
- This module provides core abstractions for handling data in retrieval-augmented generation (RAG) pipelines, vector stores, and document processing workflows.
- We can use Qdrant, or other vector databeses, through this Documents interface
- Reference: 
    - https://reference.langchain.com/python/langchain_core/documents/
    - https://docs.langchain.com/oss/python/integrations/vectorstores/qdrant#manage-vector-store

In [1]:
from datasets import load_dataset
import pandas as pd
from local_rag.core.text_cleaning import clean_text
from local_rag.core.chunking import SentenceTextSplitter


DATASET_NAME = "PrimeQA/clapnq_passages"
dataset = load_dataset(DATASET_NAME, split="train")
df = dataset.to_pandas()
df = df.head(100) # demo with first 100 rows

/home/joshuale/miniconda3/envs/local-rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Prepare the lists of texts and metadatas

In [2]:
df["doc_id"] = df["id"].str.split("_").str[0]
df_doc = df.groupby("doc_id", as_index=False).agg({"text": " ".join, "title": "first"})
df_doc["cleaned_text"] = df_doc["text"].apply(lambda x: clean_text(x))
cleaned_texts = df_doc["cleaned_text"].tolist()
titles = df_doc["title"].tolist()

sentence_splitter = SentenceTextSplitter(keep_separator="end")
documents = []

### 2. Chunk and create document obj for each chunk with metadata

In [3]:
# iteratively chunk each cleaned text and create documents for each chunk
# each document of the same text will have the same metadata
for text, title in zip(cleaned_texts, titles):
    docs = sentence_splitter.create_documents(
        texts=[text],
        metadatas=[{"title": title}]
    )
    documents.extend(docs)

In [4]:
# we can see that each text is splitted into multiple Document objects, 
# each document has a metadata with the title
print(len(documents))
documents[:10]

552


[Document(metadata={'title': 'Oh, Kay!'}, page_content='Oh, Kay!'),
 Document(metadata={'title': 'Oh, Kay!'}, page_content='1955 Studio Cast Recording Music George Gershwin Lyrics Ira Gershwin Book Guy Bolton P. G. Wodehouse Basis play La Presidente Productions 1926 Broadway 1927 West End 1928 Broadway revival 1928 Film 1960 Off-Broadway revival 1990 Broadway revival Oh, Kay!'),
 Document(metadata={'title': 'Oh, Kay!'}, page_content='is a musical with music by George Gershwin, lyrics by Ira Gershwin, and a book by Guy Bolton and P. G. Wodehouse.'),
 Document(metadata={'title': 'Oh, Kay!'}, page_content='It is based on the play La Presidente by Maurice Hanniquin and Pierre Veber.'),
 Document(metadata={'title': 'Oh, Kay!'}, page_content='The plot revolves around the adventures of the Duke of Durham and his sister, Lady Kay, English bootleggers in Prohibition Era America.'),
 Document(metadata={'title': 'Oh, Kay!'}, page_content='Kay finds herself falling in love with a man who seems una

In [5]:
# to access the page content, we can call its attribute
documents[0].page_content

'Oh, Kay!'

### 3. Combine chunk and embedding
- With the langchain_qdrant.QdrantVectorStore, we can combine 2 steps into 1: embedding texts of documents, and ingesting these documents with their vectors into the specified vector store collection
- The embedding model is wrapped with the Embeddings interface, the chunks are wrapped with the Documents interface with metadata fields. These make the process seamless

In [6]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient # note that the client is still from qdrant_client package, not langchain_qdrant
from qdrant_client.http.models import Distance, VectorParams
from src.logger import get_logger
from src.embeddings import CustomEmbeddings
from src.chunking import generate_chunk_id


logger = get_logger(__name__)
EMBEDDING_SVC_URL = "http://localhost:5002/invocations"
QDRANT_SVC_URL = "http://localhost:6333"
VECTOR_SIZE = 384 
embeddings = CustomEmbeddings(endpoint_url=EMBEDDING_SVC_URL)
client = QdrantClient(url=QDRANT_SVC_URL)

collection_names = ["clapqa_sample"]
for collection_name in collection_names:
    if client.collection_exists(collection_name=collection_name):
        logger.info(
            f"Collection {collection_name} already exists. It will be used."
        )

    else:
        logger.warning(
            f"Collection {collection_name} does not exist. Creating with specifications..."
        )
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
        )
        logger.info("Created new collection.")

# init the vector store instance:
# vector_store = QdrantVectorStore(
#     client=client,
#     collection_name=collection_name,
#     embedding=embeddings,
# )
vector_store = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    collection_name=collection_name,
    url=QDRANT_SVC_URL,
)

2025-12-13 13:40:43 INFO     Collection clapqa_sample already exists. It will be used.

In [ ]:
# create IDs for each Document (chunk) based on its content
uuids = [generate_chunk_id(chunk_text=document.page_content) for document in documents]

In [10]:
# add documents to the vector store
vector_store.add_documents(documents=documents, ids=uuids)

['82f8f07d-468a-d4cb-96cf-0a42f111168b',
 'e0425696-613f-5121-a098-f13e25199f5c',
 '027d6d30-7b72-bbfd-88ae-c24a71fea8f8',
 'e911e482-1d8b-dd90-7d7b-75aaa7c7ef7f',
 '2aa6dafd-1e7e-b05f-149c-31af7e2f0c65',
 'e229dafb-a6d0-a0b9-8f57-fe8f46089dae',
 '9122b529-7bb5-602f-38b4-28ffee4ec514',
 'b9f8b893-e11a-2f3d-9e46-a51e302a08aa',
 '78917ca7-941b-3f48-3654-8dcdb630a00c',
 '564e52ea-d417-5916-7fe1-35931d36db6b',
 '54f2d346-9c76-53a9-87c2-553ce1e11810',
 '269b1c0f-bb77-8acc-9031-de8805e74ea2',
 'b83f8ee3-b0d5-dc91-25b7-b9f2268eab40',
 '112ebfc5-2cdd-347a-5774-af7fa4424318',
 '44a35b21-4443-2af8-d9ff-9909d7d5b5f0',
 '6feaf0a9-38c7-42fe-80bf-efdcbc39366b',
 '55e5eb6e-a5f3-3a44-b9e8-f788f4a3027e',
 '7efc3fb6-1821-f9f5-5689-95ff0492b121',
 'e173e29f-c99b-7cc2-53a4-2205b7bd76ec',
 'bf70d8af-3d55-5cb3-532b-fab93ad1b00f',
 '0077deaa-2eea-349e-a9a1-2eb65c20323a',
 'b0a8ef6e-ba1e-8b98-2902-790fa25cf04a',
 '70d234cd-2521-9af0-ed2f-c698dfcffe23',
 '35a0e320-308f-a539-fb40-94562af1c190',
 'dfaf41f0-c054-

In [14]:
# testing asynchronous add
await vector_store.aadd_documents(documents=documents, ids=uuids)  # test adding again to see if duplicates are created

['82f8f07d-468a-d4cb-96cf-0a42f111168b',
 'e0425696-613f-5121-a098-f13e25199f5c',
 '027d6d30-7b72-bbfd-88ae-c24a71fea8f8',
 'e911e482-1d8b-dd90-7d7b-75aaa7c7ef7f',
 '2aa6dafd-1e7e-b05f-149c-31af7e2f0c65',
 'e229dafb-a6d0-a0b9-8f57-fe8f46089dae',
 '9122b529-7bb5-602f-38b4-28ffee4ec514',
 'b9f8b893-e11a-2f3d-9e46-a51e302a08aa',
 '78917ca7-941b-3f48-3654-8dcdb630a00c',
 '564e52ea-d417-5916-7fe1-35931d36db6b',
 '54f2d346-9c76-53a9-87c2-553ce1e11810',
 '269b1c0f-bb77-8acc-9031-de8805e74ea2',
 'b83f8ee3-b0d5-dc91-25b7-b9f2268eab40',
 '112ebfc5-2cdd-347a-5774-af7fa4424318',
 '44a35b21-4443-2af8-d9ff-9909d7d5b5f0',
 '6feaf0a9-38c7-42fe-80bf-efdcbc39366b',
 '55e5eb6e-a5f3-3a44-b9e8-f788f4a3027e',
 '7efc3fb6-1821-f9f5-5689-95ff0492b121',
 'e173e29f-c99b-7cc2-53a4-2205b7bd76ec',
 'bf70d8af-3d55-5cb3-532b-fab93ad1b00f',
 '0077deaa-2eea-349e-a9a1-2eb65c20323a',
 'b0a8ef6e-ba1e-8b98-2902-790fa25cf04a',
 '70d234cd-2521-9af0-ed2f-c698dfcffe23',
 '35a0e320-308f-a539-fb40-94562af1c190',
 'dfaf41f0-c054-